# POLVIFIDEL Pilot Study

End-to-end pipeline: download images → detect entities → generate GPT-4o captions → compute POLVIFIDEL → summarize.

**Before running:** Add `OPENAI_API_KEY` to Colab Secrets (🔑 icon in left sidebar → *Add new secret*).

In [ ]:
%%capture
!pip install insightface onnxruntime openai rouge-score
!python -m spacy download en_core_web_sm
import nltk
for pkg in ['punkt', 'punkt_tab', 'wordnet', 'averaged_perceptron_tagger', 'averaged_perceptron_tagger_eng']:
    nltk.download(pkg, quiet=True)
print('Packages ready')

In [ ]:
import base64, io, json, os, pickle, re, string, time, warnings
from collections import Counter
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import torch
from PIL import Image, ImageDraw, ImageFont
from tqdm.notebook import tqdm
from google.colab import drive, userdata

drive.mount('/content/drive')

# ── UPDATE if your Drive path differs ──────────────────────────────────────
PROJ_ROOT = Path('/content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL')
# ───────────────────────────────────────────────────────────────────────────

DATA_DIR       = PROJ_ROOT / 'data'
IMAGES_DIR     = DATA_DIR / 'images'
DETECTIONS_DIR = DATA_DIR / 'detections'
CAPTIONS_DIR   = DATA_DIR / 'captions'
METRICS_DIR    = DATA_DIR / 'metrics'
PILOT_CSV      = DATA_DIR / 'df_pilot.csv'

for d in [IMAGES_DIR, DETECTIONS_DIR, CAPTIONS_DIR, METRICS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# ── Config ─────────────────────────────────────────────────────────────────
N_PILOT      = 20
BETA         = 1.0
CONDITIONS   = ['baseline', 'textual_aux', 'visual_aux', 'annotated_aux']
GPT4O_MODEL  = 'gpt-4o'
MAX_TOKENS   = 160

POLITICAL_DESKS = {'Politics', 'National', 'Washington', 'Foreign', 'U.S.', 'World'}
GDINO_MODEL  = 'IDEA-Research/grounding-dino-tiny'
GDINO_BOX_THRESHOLD  = 0.35
GDINO_TEXT_THRESHOLD = 0.25
OBJECT_QUERIES = [
    'flag', 'banner', 'sign', 'poster', 'podium', 'microphone',
    'hat', 'badge', 'button', 'campaign shirt', 'protest sign',
]
FACE_QUALITY_THRESHOLD    = 0.5
FACE_SIMILARITY_THRESHOLD = 0.4
VIFIDEL_THRESHOLD = 0.28
CLIP_MODEL_ID = 'openai/clip-vit-base-patch32'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'Project root: {PROJ_ROOT}')

## Step 1 — Download Pilot Images

In [ ]:
def clean_image_id(url):
    stem = Path(url.split('?')[0]).stem
    return re.sub(
        r'-(jumbo|articleLarge|articleInline|master\d+|popup|'
        r'mediumThreeByTwo\d+|videoSmall|videoLarge|sfSpan|sub)$',
        '', stem,
    )

def download_image(url, dest):
    try:
        r = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=15)
        r.raise_for_status()
        if 'image' not in r.headers.get('content-type', ''):
            return False
        dest.write_bytes(r.content)
        return True
    except Exception:
        return False

df_nyt = pd.read_csv(DATA_DIR / 'df_2025.csv')
df_nyt = df_nyt.dropna(subset=['multimedia_default_url', 'caption'])
df_nyt = df_nyt[df_nyt['multimedia_default_url'].str.startswith('http')]

pool = df_nyt[df_nyt['news_desk'].isin(POLITICAL_DESKS)]
if len(pool) < N_PILOT * 3:
    pool = df_nyt
if 'web_url' in pool.columns:
    pool = pool.drop_duplicates(subset=['web_url'])
pool = pool.sample(frac=1, random_state=42).reset_index(drop=True)

rows, already_done = [], {p.stem for p in IMAGES_DIR.glob('*.jpg')}

for _, rec in tqdm(pool.iterrows(), total=len(pool), desc='Downloading'):
    if len(rows) >= N_PILOT:
        break
    url = rec['multimedia_default_url']
    image_id = clean_image_id(url)
    if image_id in already_done:
        rows.append({'image_id': image_id, 'caption': rec['caption']})
        continue
    dest = IMAGES_DIR / f'{image_id}.jpg'
    if download_image(url, dest):
        rows.append({'image_id': image_id, 'caption': rec['caption']})
        already_done.add(image_id)
    time.sleep(0.2)

df_pilot = pd.DataFrame(rows)
df_pilot.to_csv(PILOT_CSV, index=False)
print(f'Downloaded {len(df_pilot)} images')

## Step 2 — Entity Detection

Runs Grounding-DINO (objects) + InsightFace (faces). Face *identity matching* is skipped unless you have a gallery — faces are still detected and will appear as bounding boxes in the annotated_aux condition.

In [5]:
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from insightface.app import FaceAnalysis

print('Loading Grounding-DINO...')
gdino_processor = AutoProcessor.from_pretrained(GDINO_MODEL)
gdino_model = AutoModelForZeroShotObjectDetection.from_pretrained(GDINO_MODEL).to(DEVICE)
gdino_model.eval()
print('Grounding-DINO ready')

print('Loading InsightFace...')
face_app = FaceAnalysis(
    name='buffalo_l',
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider'],
)
face_app.prepare(ctx_id=0 if DEVICE == 'cuda' else -1)
print('InsightFace ready')

# Optional gallery for face identity matching
gallery = {}
gallery_path = DATA_DIR / 'gallery' / 'embeddings.pkl'
if gallery_path.exists():
    with open(gallery_path, 'rb') as f:
        gallery = pickle.load(f)
    print(f'Gallery loaded: {len(gallery)} identities')
else:
    print('No gallery — faces detected but not identified')

Loading Grounding-DINO...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

KeyboardInterrupt: 

In [ ]:
def detect_objects(image):
    query = ' . '.join(OBJECT_QUERIES) + ' .'
    inputs = gdino_processor(images=image, text=query, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = gdino_model(**inputs)
    results = gdino_processor.post_process_grounded_object_detection(
        outputs, inputs['input_ids'],
        threshold=GDINO_BOX_THRESHOLD,
        text_threshold=GDINO_TEXT_THRESHOLD,
        target_sizes=[image.size[::-1]],
    )[0]
    out = []
    for score, label, box in zip(results['scores'], results['labels'], results['boxes']):
        x0, y0, x1, y1 = box.tolist()
        out.append({'label': label, 'score': round(float(score), 3),
                    'bbox': [round(x0), round(y0), round(x1), round(y1)]})
    return out

def match_embedding(embedding):
    best_name, best_score = None, -1.0
    for name, refs in gallery.items():
        for ref in refs:
            sim = float(np.dot(embedding, ref))
            if sim > best_score:
                best_score, best_name = sim, name
    if best_score >= FACE_SIMILARITY_THRESHOLD:
        return best_name, best_score, 'matched'
    return None, best_score, 'unknown'

def detect_faces(image):
    img_bgr = np.array(image.convert('RGB'))[:, :, ::-1]
    out = []
    for face in face_app.get(img_bgr):
        x0, y0, x1, y1 = face.bbox.astype(int).tolist()
        quality = float(face.det_score)
        if quality < FACE_QUALITY_THRESHOLD:
            name, score, status = None, None, 'low_quality'
        elif gallery:
            name, score, status = match_embedding(face.embedding)
        else:
            name, score, status = None, None, 'unknown'
        out.append({'bbox': [x0, y0, x1, y1], 'det_score': round(quality, 3),
                    'name': name,
                    'match_score': round(score, 3) if score is not None else None,
                    'match_status': status})
    return out

pilot_ids = set(df_pilot['image_id'].astype(str))
image_paths = [p for p in sorted(IMAGES_DIR.glob('*.jpg')) if p.stem in pilot_ids]
done = {p.stem for p in DETECTIONS_DIR.glob('*.json')}
remaining = [p for p in image_paths if p.stem not in done]
print(f'Detecting entities in {len(remaining)} images...')

for img_path in tqdm(remaining):
    try:
        image = Image.open(img_path).convert('RGB')
        result = {
            'image_id': img_path.stem,
            'objects': detect_objects(image),
            'faces': detect_faces(image),
        }
        with open(DETECTIONS_DIR / f'{img_path.stem}.json', 'w') as f:
            json.dump(result, f, indent=2)
    except Exception as e:
        print(f'  ERROR {img_path.stem}: {e}')

print('Detection complete')

## Step 3 — Generate Captions (GPT-4o, 4 conditions)

In [8]:
CAPTION_LEN = 'Write up to 3 sentences, no more than 100 words in total.'

BASELINE_PROMPT = (
    'Describe this image. Focus on the people present, their identities '
    'if recognizable, any objects or symbols with political significance, and the '
    'relationships or interactions between people. Be specific and factual. '
    + CAPTION_LEN
)
TEXTUAL_TEMPLATE = (
    'The following political entities were automatically detected in this image:\n'
    '{entity_list}\n\n'
    'Using this information as grounding, describe the image. Focus on '
    'the people present, their identities, any objects or symbols with political '
    'significance, and the relationships or interactions between people. '
    'Be specific and factual. ' + CAPTION_LEN
)
VISUAL_PROMPT = (
    'The cropped images provided show political entities detected in the main image. '
    'Using these as grounding, describe the main image. Be specific and factual. '
    + CAPTION_LEN
)
ANNOTATED_PROMPT = (
    'The image has bounding boxes and labels marking detected political entities. '
    'Using these annotations as grounding, describe the image. Be specific and factual. '
    + CAPTION_LEN
)

def load_detection(image_id):
    p = DETECTIONS_DIR / f'{image_id}.json'
    if not p.exists():
        return {'objects': [], 'faces': []}
    with open(p) as f:
        return json.load(f)

def format_entity_list(det):
    lines = []
    matched_faces = [fc['name'] for fc in det.get('faces', [])
                     if fc.get('match_status') == 'matched' and fc.get('name')]
    objects = det.get('objects', [])
    if matched_faces:
        lines.append('Identified people: ' + ', '.join(matched_faces))
    if objects:
        counts = Counter(o['label'] for o in objects)
        lines.append('Detected objects: ' + ', '.join(
            f'{n} {lbl}' if n > 1 else lbl for lbl, n in counts.most_common()
        ))
    return '\n'.join(lines) if lines else 'No political entities detected.'

def encode_image(path):
    with open(path, 'rb') as f:
        return base64.b64encode(f.read()).decode()

def pil_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format='JPEG')
    return base64.b64encode(buf.getvalue()).decode()

def draw_annotations(img_path, det):
    img = Image.open(img_path).convert('RGB')
    draw = ImageDraw.Draw(img)
    font = ImageFont.load_default()
    for entity in det.get('faces', []) + det.get('objects', []):
        bb = entity.get('bbox')
        label = entity.get('name') or entity.get('label', '')
        if bb:
            draw.rectangle(bb, outline='red', width=2)
            draw.text((bb[0]+2, bb[1]+2), label, fill='red', font=font)
    return img

def crop_entities(img_path, det):
    img = Image.open(img_path).convert('RGB')
    crops = []
    for entity in det.get('faces', []) + det.get('objects', []):
        bb = entity.get('bbox')
        if bb:
            crops.append(img.crop(bb))
    return crops

In [ ]:
from openai import OpenAI
client = OpenAI()

def call_gpt4o(img_path, condition, det):
    content = []
    if condition == 'baseline':
        content += [{'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{encode_image(img_path)}'}},
                    {'type': 'text', 'text': BASELINE_PROMPT}]
    elif condition == 'textual_aux':
        content += [{'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{encode_image(img_path)}'}},
                    {'type': 'text', 'text': TEXTUAL_TEMPLATE.format(entity_list=format_entity_list(det))}]
    elif condition == 'visual_aux':
        content.append({'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{encode_image(img_path)}'}})
        for crop in crop_entities(img_path, det)[:5]:
            content.append({'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{pil_to_b64(crop)}'}})
        content.append({'type': 'text', 'text': VISUAL_PROMPT})
    elif condition == 'annotated_aux':
        content += [{'type': 'image_url', 'image_url': {'url': f'data:image/jpeg;base64,{pil_to_b64(draw_annotations(img_path, det))}'}},
                    {'type': 'text', 'text': ANNOTATED_PROMPT}]
    resp = client.chat.completions.create(
        model=GPT4O_MODEL,
        messages=[{'role': 'user', 'content': content}],
        max_tokens=MAX_TOKENS,
        temperature=0,
        seed=42,
    )
    return {'caption': resp.choices[0].message.content.strip(),
            'prompt_tokens': resp.usage.prompt_tokens,
            'completion_tokens': resp.usage.completion_tokens}

image_paths = [IMAGES_DIR / f"{iid}.jpg" for iid in df_pilot['image_id']
               if (IMAGES_DIR / f"{iid}.jpg").exists()]

total = len(image_paths) * len(CONDITIONS)
print(f'{total} API calls  |  ~${total * 0.01:.2f}  |  ~{total // 60 + 1} min')

for condition in CONDITIONS:
    out_path = CAPTIONS_DIR / f'gpt4o_{condition}.csv'
    done_ids = set(pd.read_csv(out_path)['image_id'].astype(str)) if out_path.exists() else set()
    remaining = [p for p in image_paths if p.stem not in done_ids]
    print(f'\n{condition}: {len(remaining)} remaining')
    for img_path in tqdm(remaining, desc=condition):
        det = load_detection(img_path.stem)
        t0 = time.time()
        try:
            res = call_gpt4o(img_path, condition, det)
            pd.DataFrame([{
                'image_id': img_path.stem, 'model': 'gpt4o', 'condition': condition,
                'caption': res['caption'],
                'prompt_tokens': res['prompt_tokens'],
                'completion_tokens': res['completion_tokens'],
                'timestamp': datetime.utcnow().isoformat(),
            }]).to_csv(out_path, mode='a', header=not out_path.exists(), index=False)
        except Exception as e:
            print(f'  ERROR {img_path.stem}: {e}')
            time.sleep(10)
            continue
        elapsed = time.time() - t0
        if elapsed < 1.0:
            time.sleep(1.0 - elapsed)

print('\nCaption generation complete')

## Step 4 — Compute Metrics

In [6]:
import spacy
from nltk.tokenize import word_tokenize
from nltk.translate.bleu_score import SmoothingFunction, sentence_bleu
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer as rouge_lib
from transformers import CLIPModel, CLIPTokenizer

nlp = spacy.load('en_core_web_sm')
clip_tok = CLIPTokenizer.from_pretrained(CLIP_MODEL_ID)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(DEVICE)
clip_model.eval()
print('NLP tools ready')

def tokenize(text):
    return word_tokenize(text.lower().translate(str.maketrans('', '', string.punctuation)))

def noun_phrases(text):
    return [c.text.lower() for c in nlp(text).noun_chunks if len(c.text) > 1]

def clip_embs(texts):
    inputs = clip_tok(texts, padding=True, truncation=True, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        out = clip_model.get_text_features(**inputs)
        # transformers 5.x returns a model output object instead of a tensor
        if not isinstance(out, torch.Tensor):
            out = out.pooler_output
            if hasattr(clip_model, 'text_projection'):
                out = clip_model.text_projection(out)
        e = out / out.norm(dim=-1, keepdim=True)
    return e.cpu().numpy()

def detected_labels(det):
    labels = [o['label'] for o in det.get('objects', [])]
    for f in det.get('faces', []):
        if f.get('match_status') == 'matched' and f.get('name'):
            labels.append(f['name'])
        elif f.get('det_score', 0) >= FACE_QUALITY_THRESHOLD:
            labels.append('person')
    return labels

def vifidel(hyp, det):
    labels = detected_labels(det)
    if not labels: return float('nan')
    phrases = noun_phrases(hyp)
    if not phrases: return 0.0
    sim = clip_embs(labels) @ clip_embs(phrases).T
    return float((sim.max(axis=1) >= VIFIDEL_THRESHOLD).sum()) / len(labels)

def hal(hyp, det):
    phrases = noun_phrases(hyp)
    if not phrases: return float('nan')
    labels = detected_labels(det)
    if not labels: return 1.0
    sim = clip_embs(phrases) @ clip_embs(labels).T
    grounded = (sim.max(axis=1) >= VIFIDEL_THRESHOLD).sum()
    return float(len(phrases) - grounded) / len(phrases)

def polvifidel(hyp, det):
    r, h = vifidel(hyp, det), hal(hyp, det)
    if np.isnan(r) or np.isnan(h): return float('nan')
    return r * (1.0 - h) ** BETA

def bleu(hyp_tok, ref_tok):
    sf = SmoothingFunction().method1
    return {
        'bleu1': sentence_bleu([ref_tok], hyp_tok, weights=(1,0,0,0), smoothing_function=sf),
        'bleu4': sentence_bleu([ref_tok], hyp_tok, weights=(.25,.25,.25,.25), smoothing_function=sf),
    }

def rouge_l(hyp, ref):
    return rouge_lib.RougeScorer(['rougeL'], use_stemmer=True).score(ref, hyp)['rougeL'].fmeasure

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

NLP tools ready


In [9]:
# Load captions
caption_dfs = [pd.read_csv(p) for p in CAPTIONS_DIR.glob('*.csv')]
captions_df = pd.concat(caption_dfs, ignore_index=True)
captions_df['image_id'] = captions_df['image_id'].astype(str)

refs = dict(zip(df_pilot['image_id'].astype(str), df_pilot['caption']))
captions_df = captions_df[captions_df['image_id'].isin(refs)]

# Keep only images with at least one detected entity
def has_entities(image_id):
    return len(detected_labels(load_detection(image_id))) > 0

before = len(captions_df)
captions_df = captions_df[captions_df['image_id'].apply(has_entities)].reset_index(drop=True)
print(f'Kept {captions_df["image_id"].nunique()} / {before // len(CONDITIONS)} images with detections')

print(f'Computing metrics for {len(captions_df)} rows...')

rows = []
for _, row in tqdm(captions_df.iterrows(), total=len(captions_df)):
    iid   = row['image_id']
    hyp   = str(row['caption'])
    ref   = refs[iid]
    det   = load_detection(iid)
    ht, rt = tokenize(hyp), tokenize(ref)
    rows.append({
        'image_id':   iid,
        'model':      row['model'],
        'condition':  row['condition'],
        **bleu(ht, rt),
        'meteor':     meteor_score([rt], ht),
        'rouge_l':    rouge_l(hyp, ref),
        'vifidel':    vifidel(hyp, det),
        'hal':        hal(hyp, det),
        'polvifidel': polvifidel(hyp, det),
    })

metrics_df = pd.DataFrame(rows)
metrics_out = METRICS_DIR / 'metrics.csv'
metrics_df.to_csv(metrics_out, index=False)
print(f'Saved → {metrics_out}')
metrics_df.groupby('condition')[['polvifidel','vifidel','hal']].mean().round(3)

Kept 20 / 20 images with detections
Computing metrics for 80 rows...


  0%|          | 0/80 [00:00<?, ?it/s]

Saved → /content/drive/MyDrive/Coursework/dissertation/POLVIFIDEL/data/metrics/metrics.csv


,polvifidel,vifidel,hal
condition,,,
annotated_aux,1.0,1.0,0.0
baseline,1.0,1.0,0.0
textual_aux,1.0,1.0,0.0
visual_aux,1.0,1.0,0.0


## Step 5 — Pilot Summary

In [10]:
from scipy import stats

FOCUS = ['polvifidel', 'vifidel', 'hal', 'bleu4', 'meteor', 'rouge_l']
ORDER = ['baseline', 'textual_aux', 'visual_aux', 'annotated_aux']

def sig(p):
    if np.isnan(p): return ''
    return '***' if p<.001 else '**' if p<.01 else '*' if p<.05 else '.' if p<.10 else ''

# ── Mean ± SD ────────────────────────────────────────────────────────────
print('=' * 72)
print('Mean ± SD by condition  (GPT-4o, n ≈ 20)')
print('=' * 72)
summary = []
for cond in ORDER:
    sub = metrics_df[metrics_df['condition'] == cond]
    row = {'condition': cond}
    for m in FOCUS:
        v = sub[m].dropna()
        row[m] = f"{v.mean():.3f}±{v.std():.3f}" if len(v) else '—'
    summary.append(row)
print(pd.DataFrame(summary).set_index('condition').to_string())

# ── Δ vs baseline ────────────────────────────────────────────────────────
print('\n' + '=' * 72)
print('Δ vs baseline  (Wilcoxon signed-rank, paired by image_id)')
print('=' * 72)
hdr = f"{'condition':<20}" + ''.join(f"{m:>14}" for m in FOCUS)
print(hdr)
print('-' * len(hdr))
for cond in ORDER:
    if cond == 'baseline': continue
    row_str = f'{cond:<20}'
    for m in FOCUS:
        if m not in metrics_df.columns:
            row_str += f"{'—':>14}"; continue
        base = metrics_df[metrics_df['condition']=='baseline'][['image_id',m]].dropna()
        comp = metrics_df[metrics_df['condition']==cond][['image_id',m]].dropna()
        paired = base.merge(comp, on='image_id', suffixes=('_b','_c'))
        if len(paired) < 5:
            row_str += f"{'n/a':>14}"; continue
        delta = paired[f'{m}_c'].mean() - paired[f'{m}_b'].mean()
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            _, pval = stats.wilcoxon(paired[f'{m}_c'], paired[f'{m}_b'])
        cell = f"{'+'if delta>=0 else ''}{delta:.3f}{sig(pval)}"
        row_str += f"{cell:>14}"
    print(row_str)

print('\n. p<.10  * p<.05  ** p<.01  *** p<.001')
print('Note: n≈20 — treat as directional signals, not confirmatory.')

Mean ± SD by condition  (GPT-4o, n ≈ 20)
                polvifidel      vifidel          hal        bleu4       meteor      rouge_l
condition                                                                                  
baseline       1.000±0.000  1.000±0.000  0.000±0.000  0.012±0.016  0.126±0.051  0.109±0.034
textual_aux    1.000±0.000  1.000±0.000  0.000±0.000  0.010±0.016  0.121±0.070  0.105±0.037
visual_aux     1.000±0.000  1.000±0.000  0.000±0.000  0.015±0.026  0.142±0.123  0.122±0.057
annotated_aux  1.000±0.000  1.000±0.000  0.000±0.000  0.013±0.017  0.139±0.080  0.118±0.039

Δ vs baseline  (Wilcoxon signed-rank, paired by image_id)
condition               polvifidel       vifidel           hal         bleu4        meteor       rouge_l
--------------------------------------------------------------------------------------------------------
textual_aux                 +0.000        +0.000        +0.000        -0.001        -0.005        -0.004
visual_aux                  +0.00

## Finish Phase 1 — classify mass cells, rebuild corpus, download

Runs the remaining Phase-1 steps from the repo's `scripts/` (the elite/figure cells are already done):

1. **`01c_classify_mass_cells.py`** — an LLM (GPT-4o-mini) reads each candidate's `headline`+`caption` and labels the mass cells by **leaning + salience**, dropping policy-article noise (~509 calls, ~$0.10).
2. **`01b_consolidate_corpus.py`** — rebuilds `data/df_corpus.csv` from the cleaned mass cells + elite cells.
3. **`download_corpus.py`** — downloads any new mass images.

Requires `OPENAI_API_KEY` in Colab Secrets. Re-running is cheap/idempotent (LLM results are cached, downloads skip existing files).

In [ ]:
# Finish Phase 1: classify mass cells (LLM) -> rebuild corpus -> download new images
# Prereqs: run the setup cell above first (mounts Drive, sets OPENAI_API_KEY, defines PROJ_ROOT).
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

SCRIPTS     = str(PROJ_ROOT / 'scripts')
RECOLLECTED = str(PROJ_ROOT / 'data' / 'recollected')

# 1.2  LLM leaning + salience for the mass cells (~509 calls, ~$0.10 on gpt-4o-mini)
#      Tip: add `--dry-run` to size the candidate pool for free; `--model gpt-4o` for max accuracy.
!cd "{SCRIPTS}" && python 01c_classify_mass_cells.py

# 1.3  rebuild data/df_corpus.csv from the cleaned mass cells + elite cells
!cd "{SCRIPTS}" && python 01b_consolidate_corpus.py --input-dir "{RECOLLECTED}"

# 1.4  download any new mass images into data/images/
!cd "{SCRIPTS}" && python download_corpus.py